# Clase 2 — Práctica Guiada
## ¿Quién se nos está yendo?
### Maestría en Fintech · ITBA · 2026

---

**Esta notebook la resolvemos juntos en clase.** Después vas a tener una segunda notebook
—la de Práctica Individual— con preguntas nuevas sobre el mismo dataset, para resolver solo.

**La pregunta de hoy:**
> *¿Qué tipo de cliente de una fintech tiene más probabilidad de irse?*

---
## ⚠️ Antes de escribir una sola línea: hacé tu propia copia

Esta notebook es **el original del curso**. Si escribís acá, se pisan entre todos y se pierde el trabajo.

**Los tres pasos, ahora:**

1. Menú **`Archivo`** → **`Guardar una copia en Drive`**
2. Se abre una pestaña nueva llamada *Copia de Clase_2_Practica_Guiada.ipynb* — **esa es tuya**
3. Hacé clic en el nombre, arriba a la izquierda, y renombrala: **`Clase2_Guiada_TuNombre`**

Cerrá la pestaña del original y trabajá siempre en la tuya.

> Es el mismo reflejo que *"Guardar como"* antes de tocar el Excel compartido del equipo.
> Tu copia queda en tu Google Drive, en la carpeta `Colab Notebooks`. No se borra cuando cerrás el navegador.

---
## Setup

No hace falta subir ningún archivo: la notebook descarga el dataset directamente desde el repositorio del curso.

Ejecutá la celda con **`Shift + Enter`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Librerías cargadas correctamente')

---
# Paso 1 · Cargar

La semana pasada vimos que una **librería** es un conjunto de herramientas que alguien ya escribió
para nosotros, y que `import pandas as pd` la carga con el apodo `pd`. Ahora la usamos por primera vez.

In [ ]:
# El dataset se descarga solo desde el repo del curso.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

df = pd.read_csv(DATOS + 'clientes.csv')

# Primeras 5 filas
df.head()

**Qué pasó acá:**

- `pd.read_csv()` leyó el archivo y lo convirtió en un **DataFrame**: la tabla de Python
- Lo guardamos en la variable `df` — convención, no obligación. Podría llamarse `clientes` o `tabla`
- `.head()` es un **método**: una acción que el objeto sabe hacer sobre sí mismo

> `objeto.método()` es la forma que van a ver el resto del curso. En Excel escribías `=PROMEDIO(A:A)`
> y le pasabas el rango a la fórmula. Acá el método va pegado al objeto. Misma idea, orden invertido.

---
# Paso 2 · Explorar

Antes de analizar nada: **¿qué hay acá adentro?** Cuántas filas, qué tipo de dato es cada columna,
si hay celdas vacías, qué rango tienen los números.

In [ ]:
# Dimensiones: (filas, columnas)
df.shape

In [ ]:
# Últimas 5 filas — para verificar que el CSV cargó completo
df.tail()

In [ ]:
# Tipos de datos y valores nulos por columna
df.info()

**Lo que nos dice `.info()`:**

| Tipo | Qué es | Ejemplo en este dataset |
|---|---|---|
| `int64` | Número entero | `edad`, `cant_productos` |
| `float64` | Número con decimales | `balance_ars` |
| `object` | Texto | `nombre`, `provincia`, `segmento` |
| `bool` | Verdadero / falso | `cliente_activo` |

Y algo más importante: la columna **`non-null count`**. Si dice 1000 en todas, no hay celdas vacías.

> En la **Clase 5** vamos a trabajar con un dataset lleno de nulos, duplicados y outliers a propósito.
> Este está limpio — es un privilegio que en la vida real casi nunca vas a tener.

In [ ]:
# Estadísticas de todas las columnas numéricas, de un vistazo
df.describe().round(2)

**Cómo se lee esta tabla:**

- `mean` → promedio · `50%` → mediana · `std` → desvío estándar · `min` / `max` → extremos

> **El truco que vale para todo el curso:** como `churn` solo tiene valores 0 y 1,
> su `mean` **es exactamente la tasa de churn de la cartera**. No hace falta contar y dividir.
> Cada vez que tengas una columna 0/1, el promedio es la proporción.

**Pregunta para el grupo:** mirando esta tabla, ¿qué te llama la atención del `balance_ars`?
Compará su `mean` contra su `50%`.

---
# Paso 3 · Seleccionar columnas

Antes de filtrar filas, quedarse con las columnas que importan.

In [ ]:
# Una columna → devuelve una Serie
df['segmento']

In [ ]:
# Varias columnas → devuelve un DataFrame (ojo el doble corchete)
df[['nombre', 'segmento', 'balance_ars']].head()

In [ ]:
# ¿Qué valores posibles tiene una columna categórica?
df['segmento'].unique()

In [ ]:
# ¿Cuántos clientes hay de cada uno?
df['segmento'].value_counts()

> Con esa última línea ya podés responderle a alguien cuántos clientes Premium tenés versus cuántos Joven.
> Sin abrir Excel, sin filtrar a mano, y de una forma que mañana se puede volver a correr igual.

---
# Paso 4 · Filtrar

En Excel el filtro es un clic. En Python es una **condición escrita**.

La lógica: la condición se evalúa fila por fila y devuelve `True` o `False`.
Python se queda solo con las filas donde dio `True`.

In [ ]:
# Clientes que se fueron (churn = 1)
df[df['churn'] == 1].head()

In [ ]:
# ¿Cuántos se fueron?
df[df['churn'] == 1].shape[0]

In [ ]:
# Guardar el filtro en una variable para reutilizarlo
se_fueron   = df[df['churn'] == 1]
se_quedaron = df[df['churn'] == 0]

print(f'Clientes que se fueron:    {len(se_fueron):>4}')
print(f'Clientes que se quedaron:  {len(se_quedaron):>4}')

> `se_fueron` ahora es **un DataFrame nuevo**. Podés seguir haciéndole preguntas solo a ese subconjunto,
> igual que le hacías preguntas al original.

### Filtros compuestos

- `&` → **Y** (ambas condiciones tienen que ser verdaderas)
- `|` → **O** (alcanza con que una lo sea)
- Los paréntesis alrededor de cada condición son **obligatorios**

> Olvidarse los paréntesis es el error de sintaxis número uno de esta clase. Si el error dice
> algo sobre *truth value* o *ambiguous*, empezá por ahí.

In [ ]:
# Clientes Premium que además están activos
df[(df['segmento'] == 'Premium') & (df['cliente_activo'] == True)].head()

In [ ]:
# Clientes con balance alto O mucha antigüedad
df[(df['balance_ars'] > 500000) | (df['antiguedad_años'] > 5)].shape[0]

**Ahora ustedes me dictan.** ¿Cómo filtramos los clientes del segmento Joven que se fueron?
Díganme la condición en palabras y la escribo.

In [ ]:
# Clientes Jóvenes que se fueron
jovenes_fugados = df[(df['segmento'] == 'Joven') & (df['churn'] == 1)]

print(f'{len(jovenes_fugados)} clientes Jóvenes se fueron')
jovenes_fugados[['nombre', 'edad', 'antiguedad_años', 'balance_ars']].head()

---
# Paso 5 · Medir

Los números solos no dicen nada. La pregunta siempre es: *¿qué significa este número para el negocio?*

In [ ]:
# Estadísticas básicas de balance
print(f"Promedio:  ${df['balance_ars'].mean():>12,.0f}")
print(f"Mediana:   ${df['balance_ars'].median():>12,.0f}")
print(f"Desvío:    ${df['balance_ars'].std():>12,.0f}")
print(f"Mínimo:    ${df['balance_ars'].min():>12,.0f}")
print(f"Máximo:    ${df['balance_ars'].max():>12,.0f}")

**Promedio vs mediana — el reflejo que quiero instalarles:**

Si el promedio es bastante mayor que la mediana, hay unos pocos clientes muy grandes tirando del promedio
hacia arriba. En ese caso *"el cliente promedio"* no describe a nadie real.

> Reportar solo el promedio cuando la distribución es asimétrica no es un error de cálculo:
> es un error de comunicación. Y en un comité, se nota.

### Agrupar es comparar

La pregunta interesante casi nunca es *¿cuánto da?*, sino ***¿cuánto da acá versus allá?***

`groupby` parte la cartera según los valores de una columna y calcula una métrica en cada parte.
Es el corazón del análisis de datos y lo vemos a fondo en la **Clase 3** — hoy alcanza con verlo funcionar.

In [ ]:
# Balance promedio: ¿quiénes se van vs quiénes se quedan?
df.groupby('churn')['balance_ars'].mean().apply(lambda x: f'${x:,.0f}')

In [ ]:
# Tasa de churn por segmento — ordenada de mayor a menor
df.groupby('segmento')['churn'].mean().sort_values(ascending=False).apply(lambda x: f'{x:.1%}')

> Eso último no es un número: es una decisión de hacia dónde va el presupuesto de retención.
> Un promedio general de churn no habilita ninguna acción. Partido por segmento, sí.

---
# Paso 6 · Ver

El objetivo no es hacer gráficos lindos. Es que el gráfico diga algo que el número solo no comunica.

**Las tres preguntas ante cualquier gráfico:**
1. ¿Qué forma tiene?
2. ¿Es lo que esperaba? Si no, ¿el dato está mal o mi supuesto estaba mal?
3. ¿Qué decisión cambia? Si ninguna, el gráfico no era necesario

In [ ]:
# Distribución de edades de la cartera
df['edad'].hist(bins=20, color='steelblue', edgecolor='white')
plt.title('Distribución de edades')
plt.xlabel('Edad')
plt.ylabel('Cantidad de clientes')
plt.show()

**¿Qué nos dice?** ¿La cartera es joven o madura? ¿Hay un rango dominante?
¿Eso es consistente con el producto que ofrece la fintech, o es un sesgo que no habíamos visto?

In [ ]:
# Clientes por segmento
df['segmento'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Clientes por segmento')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.show()

> Fijate la línea: `.value_counts()` devuelve un objeto, y a ese objeto le encadenamos `.plot()`.
> Un método que devuelve algo, que tiene sus propios métodos. Eso es la notación de punto en la práctica.

---
# Paso 7 · El caso: ¿qué le decimos al gerente de retención?

Tenemos los datos. Ahora hay que responder tres preguntas y convertirlas en una recomendación.

In [ ]:
# Pregunta 1 — ¿Cuál es la tasa de churn global?
tasa_churn = df['churn'].mean()

print(f'Tasa de churn global: {tasa_churn:.1%}')
print(f'Son {df["churn"].sum():.0f} clientes de {len(df):,}')

In [ ]:
# Pregunta 2 — ¿Qué segmento concentra la fuga?
churn_por_segmento = df.groupby('segmento')['churn'].mean().sort_values(ascending=False)

churn_por_segmento.sort_values().plot(kind='barh', color='salmon', edgecolor='white')
plt.axvline(tasa_churn, color='gray', linestyle='--', label=f'Promedio cartera ({tasa_churn:.1%})')
plt.title('Tasa de churn por segmento')
plt.xlabel('Tasa de churn')
plt.legend()
plt.show()

print(churn_por_segmento.apply(lambda x: f'{x:.1%}').to_string())

In [ ]:
# Pregunta 3 — ¿El perfil de quien se va es distinto al de quien se queda?
perfil = df.groupby('churn').agg(
    clientes=('cliente_id', 'count'),
    balance_prom=('balance_ars', 'mean'),
    edad_prom=('edad', 'mean'),
    antiguedad_prom=('antiguedad_años', 'mean'),
    productos_prom=('cant_productos', 'mean')
).round(1)
perfil.index = ['Se quedaron', 'Se fueron']

perfil

### Del análisis a la recomendación

Con esos tres números se arma el mensaje. **No es una tabla: son tres bullets.**

1. **El problema** — se fue el X% de la cartera en el período analizado
2. **El foco** — el segmento [X] fuga Y%, N veces el promedio
3. **El perfil de riesgo** — quien se va tiene $Z de balance promedio, [mayor/menor] que quien se queda

> Un análisis = **datos + interpretación + recomendación**.
> Sin el tercer componente es un reporte, y los reportes no cambian nada.

**Escribamos los tres bullets juntos, en voz alta, con los números que acabamos de sacar.**

---
## Hasta acá la práctica guiada

**Lo que hiciste hoy:** cargaste un dataset real, lo exploraste, lo filtraste, calculaste estadísticas
por grupo, hiciste dos gráficos y convertiste todo eso en una recomendación de negocio.

**Lo que sigue:** abrí la notebook **`Clase_2_Practica_Individual.ipynb`**.
Mismo dataset, preguntas nuevas, y esta vez lo resolvés vos con Gemini de copiloto.

> Recordá: primero **`Archivo → Guardar una copia en Drive`**. Siempre.